In [ ]:
import pandas as pd
import numpy as np
import pickle
!pip install optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)



df = pd.read_csv("gym_members_exercise_tracking.csv")
df.head()

# Check columns and missing values
print(df.columns)
print(df.isnull().sum())

# Standardise column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r'[\s/(]+', '_', regex=True)
    .str.strip(')')
    .str.replace(r'[^\w]', '', regex=True)
)

print('Cleaned columns:', list(df.columns))

# --- DATA CLEANING

# 1. Duplicates (harmless, good practice)
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed duplicates: {before - len(df)}")

# 2. BPM logic fixes (the real winner)
mask_avg = df['Avg_BPM'] >= df['Max_BPM']
df.loc[mask_avg, 'Avg_BPM'] = df.loc[mask_avg, 'Max_BPM'] - 5

mask_rest = df['Resting_BPM'] >= df['Avg_BPM']
df.loc[mask_rest, 'Resting_BPM'] = df.loc[mask_rest, 'Avg_BPM'] - 10

# 3. Water floor (safe, doesn't affect recovery at all)
df['Water_Intake_liters'] = df['Water_Intake_liters'].clip(lower=0.1)


# --- Features Engineering for recovery

df["Heart_Stress"] = df["Avg_BPM"] - df["Resting_BPM"] # How much heartrate increased
df["Intensity"] = df["Avg_BPM"] / df["Max_BPM"] # Percentage
df["Training_Load"] = df["Session_Duration_hours"] * df["Avg_BPM"]
df["Weekly_Load"] = df["Training_Load"] * df["Workout_Frequency_days_week"]
df["Calorie_Efficiency"] = df["Calories_Burned"] / df["Session_Duration_hours"] # Calories burned per hr

df["Recommended_Water"] = (
    df["Weight_kg"] * 0.033 +
    df["Session_Duration_hours"] * 0.7 +
    df["Calories_Burned"] / 1000
)

# This is a python dictionary that translates from a key(workout type) ----------------------------------------------
# to a value(intensity score) ------------------------------

def recovery_level(row):
    workout_factor = {
        "HIIT": 1.1,
        "Strength": 1.0,
        "Cardio": 1.0,
        "Yoga": 0.7
    }

    experience_modifier = {
        1: 1.2,
        2: 1.0,
        3: 0.8
    }

# RECOVERY SCORE

df["Workout_Factor"] = df["Workout_Type"].map({
    "Yoga": 0.7,
    "Strength": 1.0,
    "Cardio": 1.1,
    "HIIT": 1.3
})

df["Experience_Factor"] = df["Experience_Level"].map({
    1: 1.25,
    2: 1.0,
    3: 0.85
})

df["Recovery_Score"] = (
    df["Weekly_Load"] *
    (df["Intensity"] ** 1.5) *
    df["Heart_Stress"] *
    df["Workout_Factor"] *
    df["Experience_Factor"]
)


df[["Heart_Stress", "Intensity", "Training_Load", "Weekly_Load", "Recovery_Score"]].head()
df["Calories_Burned"].describe()

# Convert Recovery Score into labels: Low, Medium, High

# df["Recovery_Level"] = pd.cut(
#     df["Recovery_Score"],
#     bins=[-np.inf, 30000, 50000, np.inf],
#     labels=["Low", "Medium", "High"]
# )
df["Recovery_Level"] = pd.qcut(
    df["Recovery_Score"],
    q=3,
    labels=["Low", "Medium", "High"]
)


print("\nRECOVERY SCORE!!!!!!!!:\n", df["Recovery_Score"].describe())
print(df["Recovery_Level"].value_counts())
print(df.groupby("Recovery_Level")["Recovery_Score"].describe())

bins = pd.qcut(df["Recovery_Score"], q=3) # Checks range of each label
print(bins.value_counts())

df.groupby("Recovery_Level")[[
    "Avg_BPM",
    "Resting_BPM",
    "Session_Duration_hours",
    "Calories_Burned",
    "Workout_Frequency_days_week"
]].mean()


# Choose input features

# inputs: No water_intake nor recovery
features = [
    "Age",
    "Gender",
    "Weight_kg",
    "Height_m",
    "Max_BPM",
    "Avg_BPM",
    "Resting_BPM",
    "Session_Duration_hours",
    "Calories_Burned",
    "Workout_Type",
    "Fat_Percentage",
    "Workout_Frequency_days_week",
    "Experience_Level",
    "BMI",
    "Heart_Stress",
    "Intensity",
    "Training_Load",
    "Weekly_Load",
    "Calorie_Efficiency",
    "Workout_Factor",
    "Experience_Factor"
]

X = df[features]


# Split numeric and categorical columns

categorical_cols = ["Gender", "Workout_Type"]

numeric_cols = [col for col in features if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols), # StandardScaler(): numbers to have same range
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols) # OneHotEncoder: turns text labels to numbers
    ]
)

# ----- SPLITTING DATASETS ---------


# # --- RECOVERY ---
# Label encode target
le = LabelEncoder()
y_recovery_encoded = le.fit_transform(y_recovery)  # Low=0, Medium=1, High=2

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X, y_recovery_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_recovery_encoded
)

# Optuna
def xgb_objective(trial):
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators      = trial.suggest_int("n_estimators", 200, 800),
            learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            max_depth         = trial.suggest_int("max_depth", 3, 8),
            subsample         = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree  = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight  = trial.suggest_int("min_child_weight", 1, 10),
            reg_alpha         = trial.suggest_float("reg_alpha", 0.0, 1.0),
            reg_lambda        = trial.suggest_float("reg_lambda", 0.5, 3.0),
            eval_metric       = "mlogloss",
            random_state      = 42
        ))
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_xgb, y_train_xgb, cv=cv, scoring="accuracy", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(xgb_objective, n_trials=50)

print("Best XGB Params:", study.best_params)
print("Best XGB CV Accuracy:", round(study.best_value, 4))

# Train best XGB model (from optuna)
best_xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        **study.best_params,
        eval_metric  = "mlogloss",
        random_state = 42
    ))
])

best_xgb_model.fit(X_train_xgb, y_train_xgb)

xgb_train_preds = best_xgb_model.predict(X_train_xgb)
xgb_test_preds  = best_xgb_model.predict(X_test_xgb)

xgb_train_acc = accuracy_score(y_train_xgb, xgb_train_preds)
xgb_test_acc  = accuracy_score(y_test_xgb,  xgb_test_preds)

# decode back to Low/Medium/High for readable report
xgb_train_preds_labels = le.inverse_transform(xgb_train_preds)
xgb_test_preds_labels  = le.inverse_transform(xgb_test_preds)
y_train_labels         = le.inverse_transform(y_train_xgb)
y_test_labels          = le.inverse_transform(y_test_xgb)

print("\nRecovery Metrics")
print(f"Train Accuracy : {xgb_train_acc:.4f}")
print(classification_report(y_train_labels, xgb_train_preds_labels))
print(f"Test Accuracy  : {xgb_test_acc:.4f}")
print(classification_report(y_test_labels, xgb_test_preds_labels))




# # --- OLD RECOVERY (REPLACED WITH XGB W/ OPTUNA PARAMETERS ABOVE ---
# y_recovery = df["Recovery_Level"]

# X_train_recovery, X_test_recovery, y_train_recovery, y_test_recovery = train_test_split(
#     X,
#     y_recovery,
#     test_size=0.2,
#     random_state=42,
#     stratify=y_recovery
# )

# Build recovery classification model
# Pipeline (does multiple same time) -> preprocessing + RFC MODEL

# recovery_model = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("model", RandomForestClassifier(
#             n_estimators=300, # parameters from GridSearch
#             max_depth=12,
#             min_samples_split=3,
#             min_samples_leaf=1,
#             max_features="log2",
#             class_weight="balanced",
#             random_state=42
#         ))
#     ]
# )

# recovery_model.fit(X_train_recovery, y_train_recovery)


#Try to make: Gridsearch or Optuna

#---Gridsearch--- (commented to save runtime)
# param_grid = {
#     "model__n_estimators": [250, 300, 350, 400],
#     "model__max_depth": [8, 10, 12],
#     "model__min_samples_split": [2, 3, 4],
#     "model__min_samples_leaf": [1, 2],
#     "model__max_features": ["log2"]
# }

# grid_recovery = GridSearchCV(
#     recovery_model,
#     param_grid,
#     cv=5,
#     scoring="accuracy",
#     n_jobs=-1,
#     verbose=2
# )

# grid_recovery.fit(X_train_recovery, y_train_recovery)

# recovery_model = grid_recovery.best_estimator_

# print("Best Parameters:")
# print(grid_recovery.best_params_)

# print("Best Cross Validation Accuracy:")
# print(grid_recovery.best_score_)

# recovery_model.fit(X_train_recovery, y_train_recovery) # training


#---OPTUNA--- (produced less accuracy that gridsearch, gridsearch more effective hence we didnt use optuna)

# def objective(trial):

#     model = Pipeline(
#         steps=[
#             ("preprocessor", preprocessor),
#             ("model", RandomForestClassifier(
#                 n_estimators=trial.suggest_int("n_estimators", 100, 500),
#                 max_depth=trial.suggest_int("max_depth", 3, 15),
#                 min_samples_split=trial.suggest_int("min_samples_split", 2, 15),
#                 min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 6),
#                 max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
#                 class_weight="balanced",
#                 random_state=42
#             ))
#         ]
#     )

#     scores = cross_val_score(
#         model,
#         X_train_recovery,
#         y_train_recovery,
#         cv=3,
#         scoring="accuracy",
#         n_jobs=-1
#     )

#     return scores.mean()


# study = optuna.create_study(direction="maximize")

# study.optimize(objective, n_trials=30)

# print("Best Parameters:")
# print(study.best_params)

# print("Best Accuracy:")
# print(study.best_value)

# recovery_model = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("model", RandomForestClassifier(
#             **study.best_params,
#             class_weight="balanced",
#             random_state=42
#         ))
#     ]
# )




# y_recovery = df["Recovery_Level"]


# Testing recovery model

# recovery_train_preds = recovery_model.predict(X_train_recovery)
# recovery_test_preds = recovery_model.predict(X_test_recovery)

# train_acc = accuracy_score(y_train_recovery, recovery_train_preds)
# test_acc = accuracy_score(y_test_recovery, recovery_test_preds)

# print("\nTrain Recovery Accuracy:", train_acc)
# print(classification_report(y_train_recovery, recovery_train_preds))

# print("\nTest Recovery Accuracy:", test_acc)
# print(classification_report(y_test_recovery, recovery_test_preds))


# --- WATER INTAKE ---

y_water = df["Water_Intake_liters"] # Normal water intake

X_train_water, X_test_water, y_train_water, y_test_water = train_test_split(
    X,
    y_water,
    test_size=0.2,
    random_state=42
)

y_recommended = df["Recommended_Water"] # Recommended water intake

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_recommended, test_size=0.2, random_state=42
)

water_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            max_depth=5,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.7,
            random_state=42
        ))
    ]
)


rec_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.3,
            reg_lambda=2.0,
            random_state=42
        ))
    ]
)



water_model.fit(X_train_water, y_train_water)
rec_model.fit(X_train_r, y_train_r)


# Testing water-intake model

# Predict
water_train_preds = water_model.predict(X_train_water)
water_test_preds = water_model.predict(X_test_water)

rec_train_preds = rec_model.predict(X_train_r)
rec_test_preds = rec_model.predict(X_test_r)



# TRAIN WATER-INTAKE metrics
train_mae = mean_absolute_error(y_train_water, water_train_preds)
train_mse = mean_squared_error(y_train_water, water_train_preds)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train_water, water_train_preds)

print("\nTrain Water Metrics:")
print("MAE:", train_mae)
print("RMSE:", train_rmse)
print("R^2:", train_r2)


# TEST WATER-INTAKE metrics
test_mae = mean_absolute_error(y_test_water, water_test_preds)
test_mse = mean_squared_error(y_test_water, water_test_preds)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test_water, water_test_preds)

print("\nTest Water Metrics:")
print("MAE:", test_mae)
print("RMSE:", test_rmse)
print("R^2:", test_r2)

# TRAIN RECS metrics
train_mae_r = mean_absolute_error(y_train_r, rec_train_preds)
train_rmse_r = np.sqrt(mean_squared_error(y_train_r, rec_train_preds))
train_r2_r = r2_score(y_train_r, rec_train_preds)

print("\nRecommended Model - TRAIN")
print("MAE:", train_mae_r)
print("RMSE:", train_rmse_r)
print("R2:", train_r2_r)


# TEST RECS metrics
test_mae_r = mean_absolute_error(y_test_r, rec_test_preds)
test_rmse_r = np.sqrt(mean_squared_error(y_test_r, rec_test_preds))
test_r2_r = r2_score(y_test_r, rec_test_preds)

print("\nRecommended Model - TEST")
print("MAE:", test_mae_r)
print("RMSE:", test_rmse_r)
print("R2:", test_r2_r)



# ------ NEW UNSEEN DATA TEST -------
def prepare_user_input(user):
    user = user.copy()

    # BMI
    user["BMI"] = user["Weight_kg"] / (user["Height_m"] ** 2)

    # Engineered features
    user["Heart_Stress"] = user["Avg_BPM"] - user["Resting_BPM"]
    user["Intensity"] = user["Avg_BPM"] / user["Max_BPM"]
    user["Training_Load"] = user["Session_Duration_hours"] * user["Avg_BPM"]
    user["Weekly_Load"] = user["Training_Load"] * user["Workout_Frequency_days_week"]
    user["Calorie_Efficiency"] = user["Calories_Burned"] / user["Session_Duration_hours"]

    user["Workout_Factor"] = user["Workout_Type"].map({
    "Yoga": 0.7,
    "Strength": 1.0,
    "Cardio": 1.1,
    "HIIT": 1.3
     })

    user["Experience_Factor"] = user["Experience_Level"].map({
        1: 1.25,
        2: 1.0,
        3: 0.85
    })

    return user


new_user = pd.DataFrame({
    "Age": [20],
    "Gender": ["Female"],
    "Weight_kg": [50],
    "Height_m": [1.60],
    "Max_BPM": [90],
    "Avg_BPM": [90],
    "Resting_BPM": [90],
    "Session_Duration_hours": [2],
    "Calories_Burned": [700],
    "Workout_Type": ["Yoga"],
    "Fat_Percentage": [15],
    "Workout_Frequency_days_week": [2],
    "Experience_Level": [2]
})

new_user_ready = prepare_user_input(new_user) # Auto calculates other/new features
print("RECOVERY SCORE:",
    new_user_ready["Weekly_Load"].iloc[0] *
    (new_user_ready["Intensity"].iloc[0] ** 1.5) *
    new_user_ready["Heart_Stress"].iloc[0] *
    new_user_ready["Workout_Factor"].iloc[0] *
    new_user_ready["Experience_Factor"].iloc[0]
)
print("Recovery:", recovery_model.predict(new_user_ready)[0])
print("Water:", round(water_model.predict(new_user_ready)[0], 2), "liters")
print("Recommended Water:", round(rec_model.predict(new_user_ready)[0], 2), "liters")





Index(['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM',
       'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned',
       'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)',
       'Workout_Frequency (days/week)', 'Experience_Level', 'BMI'],
      dtype='object')
Age                              0
Gender                           0
Weight (kg)                      0
Height (m)                       0
Max_BPM                          0
Avg_BPM                          0
Resting_BPM                      0
Session_Duration (hours)         0
Calories_Burned                  0
Workout_Type                     0
Fat_Percentage                   0
Water_Intake (liters)            0
Workout_Frequency (days/week)    0
Experience_Level                 0
BMI                              0
dtype: int64
Cleaned columns: ['Age', 'Gender', 'Weight_kg', 'Height_m', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration_hours', 'Calories_Burned', 'Workout_Type', 'Fat

/tmp/ipykernel_11144/2239347835.py:134: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("Recovery_Level")["Recovery_Score"].describe())
/tmp/ipykernel_11144/2239347835.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Recovery_Level")[[


In [6]:
# Check what your actual model predicts with the test data
test_data = pd.DataFrame({
    "Age": [20],
    "Gender": ["Female"],
    "Weight_kg": [50],
    "Height_m": [1.60],
    "Max_BPM": [90],
    "Avg_BPM": [90],
    "Resting_BPM": [90],
    "Session_Duration_hours": [2],
    "Calories_Burned": [700],
    "Workout_Type": ["Yoga"],
    "Fat_Percentage": [15],
    "Workout_Frequency_days_week": [2],
    "Experience_Level": [2]
})

# Prepare features (same as training)
test_prepared = prepare_user_input(test_data)

# Get predictions from each model
recovery = recovery_model.predict(test_prepared)[0]
water = water_model.predict(test_prepared)[0]
recommended = rec_model.predict(test_prepared)[0]

print(f"Recovery: {recovery}")
print(f"Water Intake: {water}")
print(f"Recommended Water: {recommended}")

# Also print the features to see what's being sent
print("\nFeatures used:")
print(test_prepared)

NameError: name 'prepare_user_input' is not defined

In [ ]:
#--------------------------------------------------------------------
print("------------------------------------------------------------")
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = ConfusionMatrixDisplay.from_predictions(
    y_test_recovery,
    recovery_test_preds
)

plt.title("Recovery Level Prediction Confusion Matrix")
plt.show()

df["Recovery_Level"].value_counts().plot(kind="bar")

plt.title("Distribution of Recovery Levels")
plt.xlabel("Recovery Category")
plt.ylabel("Number of Samples")
plt.show()

plt.scatter(y_test_water, water_test_preds)

plt.xlabel("Actual Water Intake (liters)")
plt.ylabel("Predicted Water Intake (liters)")
plt.title("Actual vs Predicted Water Intake")

plt.plot(
    [y_test_water.min(), y_test_water.max()],
    [y_test_water.min(), y_test_water.max()],
)

plt.show()

plt.scatter(y_test_r, rec_test_preds)

plt.xlabel("Actual Recommended Water")
plt.ylabel("Predicted Recommended Water")
plt.title("Recommended Water Prediction")

plt.plot(
    [y_test_r.min(), y_test_r.max()],
    [y_test_r.min(), y_test_r.max()],
)

plt.show()

model = water_model.named_steps["model"]

cat_features = water_model.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_cols)
all_features = numeric_cols + list(cat_features)

importance_df = pd.DataFrame({
    "Feature": all_features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False).head(10)

plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.title("Top Features Influencing Water Intake Prediction")
plt.gca().invert_yaxis()
plt.show()

#----------------------------------------------------------------
pickle.dump(recovery_model, open('recovery_model.pkl', 'wb'))
pickle.dump(water_model, open('water_model.pkl', 'wb'))
pickle.dump(rec_model, open('recommended_model.pkl', 'wb'))

Index(['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM',
       'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned',
       'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)',
       'Workout_Frequency (days/week)', 'Experience_Level', 'BMI'],
      dtype='object')
Age                              0
Gender                           0
Weight (kg)                      0
Height (m)                       0
Max_BPM                          0
Avg_BPM                          0
Resting_BPM                      0
Session_Duration (hours)         0
Calories_Burned                  0
Workout_Type                     0
Fat_Percentage                   0
Water_Intake (liters)            0
Workout_Frequency (days/week)    0
Experience_Level                 0
BMI                              0
dtype: int64
Cleaned columns: ['Age', 'Gender', 'Weight_kg', 'Height_m', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration_hours', 'Calories_Burned', 'Workout_Type', 'Fat

/tmp/ipykernel_11144/3946041604.py:130: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("Recovery_Level")["Recovery_Score"].describe())
/tmp/ipykernel_11144/3946041604.py:135: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Recovery_Level")[[


KeyboardInterrupt: 